In [1]:
import glob
import nltk
nltk.download('popular', download_dir="nltk_data")
from nltk.corpus import stopwords
from nltk import word_tokenize
import string
from collections import Counter
import numpy as np
from collections import OrderedDict


[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to nltk_data...
[nltk_data]    |   Unzipping corpora\cmudict.zip.
[nltk_data]    | Downloading package gazetteers to nltk_data...
[nltk_data]    |   Unzipping corpora\gazetteers.zip.
[nltk_data]    | Downloading package genesis to nltk_data...
[nltk_data]    |   Unzipping corpora\genesis.zip.
[nltk_data]    | Downloading package gutenberg to nltk_data...
[nltk_data]    |   Unzipping corpora\gutenberg.zip.
[nltk_data]    | Downloading package inaugural to nltk_data...
[nltk_data]    |   Unzipping corpora\inaugural.zip.
[nltk_data]    | Downloading package movie_reviews to nltk_data...
[nltk_data]    |   Unzipping corpora\movie_reviews.zip.
[nltk_data]    | Downloading package names to nltk_data...
[nltk_data]    |   Unzipping corpora\names.zip.
[nltk_data]    | Downloading package shakespeare to nltk_data...
[nltk_data]    |   Unzipping corpora\shakespeare.zip.
[nltk_data]    | Do

In [11]:
def give_path(fld_path): 
    dic = {}
    file_names = glob.glob(fld_path, recursive=True)
    files_150 = file_names[0:10]
    for file in files_150:
        name = file.split('/')[-1]
        with open(file, 'r', errors='ignore') as f:
            data = f.read()
        dic[name] = data
    return dic

def wordList_removePuncs(doc_dict):
    stop = stopwords.words('english') + list(string.punctuation) + ['\n']
    wordList = []
    for doc in doc_dict.values():
        for word in word_tokenize(doc.lower().strip()):
            if not word in stop:
                wordList.append(word)
    return wordList

def termFrequencyInDoc(vocab, doc_dict):
    tf_docs = {}
    for doc_id in doc_dict.keys():
        tf_docs[doc_id] = {}
    for word in vocab:
        for doc_id, doc in doc_dict.items():
            tf_docs[doc_id][word] = doc.count(word)
    return tf_docs

def wordDocFre(vocab, doc_dict):
    df = {}
    for word in vocab:
        frq = 0
        for doc in doc_dict.values():
            if word in word_tokenize(doc.lower().strip()):
                frq = frq + 1
        df[word] = frq
    return df

def inverseDocFre(vocab, doc_fre, length):
    idf = {}
    for word in vocab:
        idf[word] = np.log2((length + 1) / doc_fre[word])
    return idf


def tfidf(vocab, tf, idf_scr, doc_dict):
    tf_idf_scr = {}
    for doc_id in doc_dict.keys():
        tf_idf_scr[doc_id] = {}
    for word in vocab:
        for doc_id, doc in doc_dict.items():
            tf_idf_scr[doc_id][word] = tf[doc_id][word] * idf_scr[word]
    return tf_idf_scr

def vectorSpaceModel(query, doc_dict, tfidf_scr):
    query_vocab = []
    for word in query.split():
        if word not in query_vocab:
            query_vocab.append(word)
    query_wc = {}
    for word in query_vocab:
        query_wc[word] = query.lower().split().count(word)
    relevance_scores = {}
    for doc_id in doc_dict.keys():
        score = 0
        for word in query_vocab:
            # Strictly use the provided logic; if a query term isn't in vocab, it contributes zero.
            if word in tfidf_scr[doc_id]:
                score += query_wc[word] * tfidf_scr[doc_id][word]
        relevance_scores[doc_id] = score
    sorted_value = OrderedDict(sorted(relevance_scores.items(), key=lambda x: x[1], reverse=True))
    top_5 = {k: sorted_value[k] for k in list(sorted_value)[:5]}
    return top_5


In [12]:
docs = give_path("./nltk_data/**/*.txt")
M = len(docs)
w_List = wordList_removePuncs(docs)
vocab = list(set(w_List))
tf_dict = termFrequencyInDoc(vocab, docs)
df_dict = wordDocFre(vocab, docs)
idf_dict = inverseDocFre(vocab, df_dict, M)
tf_idf = tfidf(vocab, tf_dict, idf_dict, docs)

print("Documents loaded:", M)
print("Vocabulary size:", len(vocab))
print("Sample documents:", list(docs.keys())[:5])


Documents loaded: 10
Vocabulary size: 3479
Sample documents: ['nltk_data\\corpora\\gazetteers\\caprovinces.txt', 'nltk_data\\corpora\\gazetteers\\countries.txt', 'nltk_data\\corpora\\gazetteers\\isocountries.txt', 'nltk_data\\corpora\\gazetteers\\LICENSE.txt', 'nltk_data\\corpora\\gazetteers\\mexstates.txt']


In [13]:
queries = [
    'Text Mining',
    'LDA',
    'topic modeling',
    'Natural language Processing',
    'generative models'
]

for i, q in enumerate(queries, 1):
    res = vectorSpaceModel(q, docs, tf_idf)
    print(f"Top 5 Documents for Query {i} ({q}):")
    for k,v in res.items():
        print(f"  {k:30s}  score={v:.3f}")
    print("-"*60)


Top 5 Documents for Query 1 (Text Mining):
  nltk_data\corpora\gazetteers\caprovinces.txt  score=0.000
  nltk_data\corpora\gazetteers\countries.txt  score=0.000
  nltk_data\corpora\gazetteers\isocountries.txt  score=0.000
  nltk_data\corpora\gazetteers\LICENSE.txt  score=0.000
  nltk_data\corpora\gazetteers\mexstates.txt  score=0.000
------------------------------------------------------------
Top 5 Documents for Query 2 (LDA):
  nltk_data\corpora\gazetteers\caprovinces.txt  score=0.000
  nltk_data\corpora\gazetteers\countries.txt  score=0.000
  nltk_data\corpora\gazetteers\isocountries.txt  score=0.000
  nltk_data\corpora\gazetteers\LICENSE.txt  score=0.000
  nltk_data\corpora\gazetteers\mexstates.txt  score=0.000
------------------------------------------------------------
Top 5 Documents for Query 3 (topic modeling):
  nltk_data\corpora\gazetteers\caprovinces.txt  score=0.000
  nltk_data\corpora\gazetteers\countries.txt  score=0.000
  nltk_data\corpora\gazetteers\isocountries.txt  s